In [1]:
import os
import numpy as np
import matplotlib.pyplot as pl

In [2]:
os.chdir('C:\cygwin64\home\codes\dispatch2\data\eos\Tomida+Hori_2016')
import EOS
from scaling import cgs

In [3]:
eos=EOS.eos_t()

reading data from table tomida.dat


The simplest integration of the hydrostatic equations for small radii, where pressure, density, and temperature are nearly constant, is to use

$$\frac{dM}{dR} = 4\pi R^2  \rho$$

$$\frac{d\ln M}{d\ln R} = 4\pi R^3 \rho/M(R) \approx 3 ,$$

which is nearly constant for small radii, and then keep track of the decrease in pressure with

$$-\frac{d\ln P}{d R^2}= -\frac{d \ln P}{2 R dR} = \frac{g\rho}{2 R P} = \frac{G M(R) \rho}{ 2 R^3 P} = F(R) \rightarrow \frac{2\pi\rho_0^2 G}{3 P_0}=F_0,$$

which is also nearly constant for small radii. A very good approximation over the first pressure interval along the adiabat is thus to assume this expression to be constant and equal to the initial value, allowing one to compute parapolic decline in pressure from zero radius to the first radius point, at $R_1 = \sqrt{-\Delta \ln P /F_0)}$

Inverting that relation gives a useful relation for larger radii,

$$-\frac{d R^2}{d\ln P} =\frac{2 R^3 P}{G M(R) \rho},$$

which allows taking given steps in $\ln P$, and computing the corresponding change in radius (squared), while (from the adiabat) knowing the RHS at given points.  This is convenient when following an adiabat.

An implementation follow below:

In [24]:
import save_restore as sr
adiabats=sr.restore('Jupiter1.save')
key='T100d-12'
v=adiabats[key]
v.keys()

dict_keys(['T', 'P', 'U', 'd', 'M', 'R', 'i0'])

Startup

In [61]:
T=v['T']
P=v['P']
M=v['M']
R=v['R']
d=v['d']
i0=v['i0']
i=i0-1
T0=T[i]
d0=d[i]
P0=P[i]
F0=2*np.pi*cgs.grav*d0**2/(3*P0)
dlnP=np.log(P[i]/P[i-1])
R[i]=0
i=i-1
R[i]=np.sqrt(dlnP/F0)
M[i]=4*np.pi*R[i]**3/3*d0
R[i]/cgs.r_earth

33.365871737518397

Integration:

In [26]:
i=i0-2
while i>0:
    i=i-1
    M[i]=M[i+1]
    R[i]=R[i+1]
    for iter in range(5):
        F=0.5*cgs.grav/2.0*(M[i  ]*d[i  ]/(P[i  ]*R[i  ]**3)+
                            M[i+1]*d[i+1]/(P[i+1]*R[i+1]**3))
        dlnP=np.log(P[i+1]/P[i])
        R[i]=np.sqrt(R[i+1]**2+dlnP/F)
        M[i]=M[i+1]+(R[i]**3-R[i+1]**3)*4*np.pi/3*(d[i]+d[i+1])/2
print(i,iter,dlnP,F,T[i])

0 4 0.0333631224981 8.36225379082e-26 100.0


Now we have good values in the range `R[0:i0-1]`, and we can add some radius and mass points above that, for good measure:

In [27]:
dlnR=0.4
i1=i0+10
for i in range(i0-1,i1):
    R[i]=R[i-1]*np.exp(-dlnR)
    M[i]=4*np.pi*R[i]**3/3*d0
    d[i]=d[i0-1]
    P[i]=P[i0-1]
    T[i]=T[i0-1]

Plot M(R)

In [39]:
pl.figure(11); pl.clf()
pl.xlabel('R [R_E]')
pl.ylabel('M [M_E]')
pl.semilogx(R[i0-1]/cgs.r_earth,M[i0-1]/cgs.m_earth,'o')
pl.semilogx(R[0:i1]/cgs.r_earth,M[0:i1]/cgs.m_earth,'-+')
pl.figure(12); pl.clf()
dMdlnR=np.gradient(M[0:i1])/np.gradient(np.log(R[0:i1]))
pl.xlabel('R [R_E]')
pl.ylabel('dM/dlnR')
pl.semilogx(R[0:i1]/cgs.r_earth,dMdlnR/cgs.m_earth);

Check BC values

In [29]:
245*5.2*1000**0.3333,R[0]/cgs.r_earth,M[0]/cgs.m_earth

(12737.066844296514, 14196.406522790121, 1091.7042464365757)

Compute optical depth:

In [62]:
kappa=0.1
tau=np.zeros(len(d))
for i in range(1,len(d)):
    tau[i]=tau[i-1]+kappa*0.5*(d[i]+d[i-1])*(R[i-1]-R[i])
    if tau[i]<1:
        i1=i
tau[0]=0.5*tau[1]
i1

95

In [63]:
pl.figure()
pl.loglog(R[0:i0]/cgs.r_earth,np.ones(i0))
pl.loglog(R[0:i0]/cgs.r_earth,tau[0:i0]);

The optical depth is such that these structures would be large hidden inside the optically thick disk that corresponds to MMSN values